In [0]:
# Databricks notebook source

dbutils.widgets.text("config_path", "")
config_path = dbutils.widgets.get("config_path")

if not config_path or config_path.strip() == "":
    raise ValueError("config_path is required")

print(f"CONFIG PATH: {config_path}")

import json
from delta.tables import DeltaTable
from pyspark.sql import functions as F

# ---------------------------------------
# LOAD CONFIG
# ---------------------------------------
config_text = "\n".join(
    row.value for row in spark.read.text(config_path).collect()
)

config = json.loads(config_text)

source_name = config["source_name"]
entity_name = config["entity_name"]
primary_key = config["primary_key"]
silver_path = config["silver_path"]
gold_path = config["gold_path"]

print("Loaded config for:", source_name)
print("Entity name:", entity_name)
print("Primary key:", primary_key)
print("Silver path:", silver_path)
print("Gold path:", gold_path)

# ---------------------------------------
# READ SILVER
# ---------------------------------------
silver_df = spark.read.format("delta").load(silver_path)

print("Silver rows:", silver_df.count())
display(silver_df)

upsert_df = silver_df.filter(F.col("op").isin("I", "U"))
delete_df = silver_df.filter(F.col("op") == "D")

print("Upsert rows:", upsert_df.count())
print("Delete rows:", delete_df.count())

# ---------------------------------------
# RESET GOLD FOR CLEAN TEST
# ---------------------------------------
dbutils.fs.rm(gold_path, True)

# build Gold schema ONLY from upsert rows
gold_business_cols = [c for c in silver_df.columns if c != "op"]

empty_gold_df = upsert_df.select(*gold_business_cols).limit(0)
empty_gold_df.write.format("delta").mode("overwrite").save(gold_path)

gold_table = DeltaTable.forPath(spark, gold_path)

# ---------------------------------------
# APPLY UPSERTS
# ---------------------------------------
if upsert_df.count() > 0:
    source_upsert_df = upsert_df.select(*gold_business_cols)

    update_map = {c: f"s.{c}" for c in gold_business_cols}
    insert_map = {c: f"s.{c}" for c in gold_business_cols}

    (
        gold_table.alias("t")
        .merge(
            source_upsert_df.alias("s"),
            f"t.{primary_key} = s.{primary_key}"
        )
        .whenMatchedUpdate(set=update_map)
        .whenNotMatchedInsert(values=insert_map)
        .execute()
    )
    print("Upsert merge completed")
else:
    print("No upsert rows to process")

gold_table = DeltaTable.forPath(spark, gold_path)

# ---------------------------------------
# APPLY DELETES
# ---------------------------------------
if delete_df.count() > 0:
    (
        gold_table.alias("t")
        .merge(
            delete_df.select(primary_key).alias("s"),
            f"t.{primary_key} = s.{primary_key}"
        )
        .whenMatchedDelete()
        .execute()
    )
    print("Delete merge completed")
else:
    print("No delete rows to process")

# ---------------------------------------
# FINAL CHECK
# ---------------------------------------
final_gold_df = spark.read.format("delta").load(gold_path)

print("FINAL GOLD COUNT =", final_gold_df.count())
final_gold_df.printSchema()
display(final_gold_df)
gold_check_df = spark.read.format("delta").load(gold_path)

print("FINAL GOLD COUNT =", gold_check_df.count())
print("FINAL GOLD COLUMNS =", gold_check_df.columns)
gold_check_df.printSchema()
display(gold_check_df)

silver_check_df = spark.read.format("delta").load(silver_path)

print("FINAL SILVER COUNT =", silver_check_df.count())
print("FINAL SILVER COLUMNS =", silver_check_df.columns)
display(silver_check_df)
dbutils.notebook.exit("GOLD SUCCESS")

In [0]:
display(
    spark.read.format("delta")
    .load("/Volumes/dbx_dev/stratus/eventhub_volume/gold/customer/")
)

event_id,entity,source_ts,customer_id,customer_name,email,updated_at,json_str,partition,offset,event_timestamp,ingest_ts,entity_name,source_name


In [0]:
dbutils.fs.rm("/Volumes/dbx_dev/stratus/eventhub_volume/gold/customer/", True)

True

In [0]:
display(
    spark.read.format("delta")
    .load("/Volumes/dbx_dev/stratus/eventhub_volume/gold/orders/")
)

event_id,entity,source_ts,order_id,customer_id,order_amount,order_status,updated_at,json_str,partition,offset,event_timestamp,ingest_ts,entity_name,source_name
o11,orders,2026-04-13,301,101,700,updated,2026-04-13,"{""event_id"": ""o11"", ""op"": ""U"", ""entity"": ""orders"", ""order_id"": 301, ""customer_id"": 101, ""order_amount"": ""700"", ""order_status"": ""updated"", ""updated_at"": ""2026-04-13"", ""source_ts"": ""2026-04-13""}",1,6,2026-04-13T15:27:04.813Z,2026-04-13T15:27:30.275Z,orders,eventhub_orders


In [0]:
dbutils.fs.rm("/Volumes/dbx_dev/stratus/eventhub_volume/gold/customer/", True)

True

In [0]:
print(config["gold_path"])

/Volumes/dbx_dev/stratus/eventhub_volume/gold/orders/


In [0]:
dbutils.fs.rm("/Volumes/dbx_dev/stratus/eventhub_volume/gold/customer/", True)

True

In [0]:
dbutils.fs.rm("/Volumes/dbx_dev/stratus/eventhub_volume/dlq/customer/", True)

True

In [0]:
dbutils.fs.rm("/Volumes/dbx_dev/stratus/eventhub_volume/gold/orders/", True)

False

In [0]:
dbutils.fs.rm("/Volumes/dbx_dev/stratus/eventhub_volume/checkpoints/orders/", recurse=True)
print("Cleared!")

Cleared!


In [0]:
dbutils.fs.rm("/Volumes/dbx_dev/stratus/eventhub_volume/customer_cdc/gold/", True)

True

In [0]:
dbutils.fs.rm("/Volumes/dbx_dev/stratus/eventhub_volume/gold/customer/", True)

False

In [0]:
dbutils.fs.rm("/Volumes/dbx_dev/stratus/eventhub_volume/gold/customer/", True)

False

In [0]:
display(dbutils.fs.ls("/Volumes/dbx_dev/stratus/eventhub_volume/gold/"))

path,name,size,modificationTime
dbfs:/Volumes/dbx_dev/stratus/eventhub_volume/gold/_delta_log/,_delta_log/,0,1775484653000
dbfs:/Volumes/dbx_dev/stratus/eventhub_volume/gold/part-00000-5fab5944-54fe-4124-a424-d8da54b069e9-c000.snappy.parquet,part-00000-5fab5944-54fe-4124-a424-d8da54b069e9-c000.snappy.parquet,3188,1775484654000


In [0]:
display(dbutils.fs.ls("/Volumes/dbx_dev/stratus/eventhub_volume/gold/"))

path,name,size,modificationTime
dbfs:/Volumes/dbx_dev/stratus/eventhub_volume/gold/_delta_log/,_delta_log/,0,1775484653000
dbfs:/Volumes/dbx_dev/stratus/eventhub_volume/gold/part-00000-5fab5944-54fe-4124-a424-d8da54b069e9-c000.snappy.parquet,part-00000-5fab5944-54fe-4124-a424-d8da54b069e9-c000.snappy.parquet,3188,1775484654000


In [0]:
dbutils.fs.rm("/Volumes/dbx_dev/stratus/eventhub_volume/gold/", True)

True

In [0]:
dbutils.fs.rm(gold_path, True)

empty_df = silver_df.filter("1 = 0").drop("op")
empty_df.write.format("delta").mode("overwrite").save(gold_path)

test_gold_df = spark.read.format("delta").load(gold_path)
print("AFTER RESET GOLD COUNT =", test_gold_df.count())
display(test_gold_df)